# 03 · Dynamic TRACER - Continual Learning on Banking77

This notebook simulates a **production deployment scenario** where Banking77 customer-support queries arrive in daily batches.

The workflow:
1. Split the Banking77 training set into **5 days** of traces
2. **Day 1** → `tracer.fit()` cold-start
3. **Days 2–5** → `tracer.update()` incremental refit
4. Track **coverage growth** and **teacher-agreement** at each checkpoint
5. Inspect **per-intent temporal deltas** - which intents become self-serve fastest?
6. Evaluate against the held-out test set at every checkpoint
7. Snapshot the **XAI report** from the final checkpoint

> **Static vs Dynamic**: `02-static-tracer` fits once on the full dataset. This notebook
> shows the *flywheel*: each new day's traces make the router smarter.

**Prerequisites**: Precomputed BGE-M3 embeddings are downloaded automatically from [adamrida/tracer-banking77](https://huggingface.co/datasets/adamrida/tracer-banking77). No API keys required.

In [ ]:
# Make tracer importable without pip install
import sys
from pathlib import Path

_src = (Path('..') / 'src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print(f'tracer source: {_src}')

# -- Download precomputed data from HuggingFace if available --
try:
    from huggingface_hub import hf_hub_download
    import shutil
    HF_REPO = "adamrida/tracer-banking77"
    _data = Path('data')
    _data.mkdir(exist_ok=True)

    _hf_files = {
        "banking77_traces.jsonl": _data / "banking77_traces.jsonl",
        "banking77_embeddings.npy": _data / "banking77_traces.npy",
        "banking77_test_embeddings.npy": _data / "banking77_test_embeddings.npy",
    }
    for hf_name, local_path in _hf_files.items():
        if not local_path.exists():
            dl = hf_hub_download(HF_REPO, hf_name, repo_type="dataset")
            shutil.copy(dl, local_path)
            print(f'  Downloaded {hf_name} -> {local_path}')
        else:
            print(f'  Exists: {local_path}')
    print('Precomputed data ready.')
except ImportError:
    print('huggingface_hub not installed. Run: pip install huggingface_hub')

In [ ]:
import json
import os
import time
from collections import defaultdict

import numpy as np

import tracer
from tracer.traces.loader import load_traces, save_traces
from tracer.types import TraceDataset

print('tracer', tracer.__version__ if hasattr(tracer, '__version__') else '0.1.0')

## Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR     = Path('data')
ARTIFACT_DIR = DATA_DIR / '.tracer_dynamic'
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Simulation parameters ─────────────────────────────────────────────────────
N_DAYS      = 5      # number of incremental batches
RANDOM_SEED = 42

print(f'Data dir  : {DATA_DIR}')
print(f'Artifacts : {ARTIFACT_DIR}')
print(f'Days      : {N_DAYS}')

## 1 · Load Banking77 & split into daily batches

In [ ]:
from datasets import load_dataset

ds = load_dataset('PolyAI/banking77')
train_ds = ds['train']
test_ds  = ds['test']

# Map integer labels → string names
label_names = train_ds.features['label'].names

train_texts  = [ex['text']           for ex in train_ds]
train_labels = [label_names[ex['label']] for ex in train_ds]
test_texts   = [ex['text']           for ex in test_ds]
test_labels  = [label_names[ex['label']] for ex in test_ds]

print(f'Train: {len(train_texts):,} samples  |  Test: {len(test_texts):,} samples')
print(f'Intents: {len(label_names)}')

In [ ]:
# Shuffle and split train into N_DAYS equal-ish chunks
rng = np.random.RandomState(RANDOM_SEED)
idx = rng.permutation(len(train_texts))

day_indices = np.array_split(idx, N_DAYS)

for d, chunk in enumerate(day_indices, 1):
    print(f'  Day {d}: {len(chunk):,} traces')

print(f'  Test : {len(test_texts):,} traces')

## 2 · Write trace JSONL files for each day

In [ ]:
def write_jsonl(texts, labels, path, ids=None):
    """Persist texts + labels as a TRACER-format JSONL trace file."""
    path = Path(path)
    with path.open('w', encoding='utf-8') as f:
        for i, (text, label) in enumerate(zip(texts, labels)):
            row = {
                'input':        text,
                'teacher':      label,
                'ground_truth': label,
                'id': str(ids[i]) if ids is not None else str(i),
            }
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


# Write per-day JSONL files (skip if already exist)
day_trace_paths = []
for d, chunk in enumerate(day_indices, 1):
    p = DATA_DIR / f'banking77_day{d}_traces.jsonl'
    if not p.exists():
        texts_d  = [train_texts[i]  for i in chunk]
        labels_d = [train_labels[i] for i in chunk]
        write_jsonl(texts_d, labels_d, p, ids=chunk)
        print(f'  Wrote {p.name}  ({len(chunk):,} traces)')
    else:
        print(f'  Exists: {p.name}')
    day_trace_paths.append(p)

# Write test JSONL
test_trace_path = DATA_DIR / 'banking77_test_traces.jsonl'
if not test_trace_path.exists():
    write_jsonl(test_texts, test_labels, test_trace_path)
    print(f'  Wrote {test_trace_path.name}')
else:
    print(f'  Exists: {test_trace_path.name}')

## 3 · Load precomputed BGE-M3 embeddings

The full training set embeddings were downloaded from HuggingFace in the setup cell. Here we split them into per-day batches to simulate the incremental arrival of traces.

In [ ]:
# Load full training embeddings from the HF-downloaded file and split into per-day chunks
_all_train_embs = np.load(DATA_DIR / 'banking77_traces.npy')
assert len(_all_train_embs) == len(train_texts), \
    f'Embedding count mismatch: {len(_all_train_embs)} vs {len(train_texts)} texts'
print(f'Loaded full training embeddings: {_all_train_embs.shape}')

In [ ]:
# Split precomputed embeddings into per-day chunks (matching the trace splits)
day_emb_paths = []
for d, chunk in enumerate(day_indices, 1):
    emb_path = DATA_DIR / f'banking77_day{d}_embeddings.npy'
    if not emb_path.exists():
        day_embs = _all_train_embs[chunk]
        np.save(emb_path, day_embs)
        print(f'  Saved day {d}: {day_embs.shape}')
    else:
        print(f'  Exists: {emb_path.name}')
    day_emb_paths.append(emb_path)

# Load test embeddings
test_emb_path = DATA_DIR / 'banking77_test_embeddings.npy'
assert test_emb_path.exists(), f'Missing {test_emb_path} -- run the setup cell.'
test_embeddings = np.load(test_emb_path)
print(f'  Test: {test_embeddings.shape}')

## 4 · Continual learning loop

- **Day 1** cold-starts with `tracer.fit()`
- After fit we copy the day-1 traces to `all_traces.jsonl` inside the artifact dir - this seeds the incremental history that `tracer.update()` reads
- **Days 2–5** call `tracer.update()`, which loads historical embeddings from the FAISS index, appends new ones, and re-fits

In [ ]:
import tracer as _tracer

checkpoints = []  # list of ArtifactManifest snapshots

# ── Day 1 : cold start ────────────────────────────────────────────────────────
print('=== Day 1 : fit ===')
result1 = _tracer.fit(
    day_trace_paths[0],
    artifact_dir=ARTIFACT_DIR,
    embeddings=np.load(day_emb_paths[0]),
)
checkpoints.append(result1.manifest)
for note in result1.notes:
    print(' ', note)
print(f'  n_traces={result1.manifest.n_traces}  '
      f'coverage={result1.manifest.coverage_cal:.1%}  '
      f'method={result1.manifest.selected_method}')

# Seed all_traces.jsonl so tracer.update() has full history from day 1 onward
_day1_ds = load_traces(day_trace_paths[0])
save_traces(_day1_ds, ARTIFACT_DIR / 'all_traces.jsonl')
print('  → seeded all_traces.jsonl with day-1 records')

In [ ]:
# ── Days 2–N : incremental updates ───────────────────────────────────────────
for d in range(2, N_DAYS + 1):
    print(f'=== Day {d} : update ===')
    result = _tracer.update(
        day_trace_paths[d - 1],
        artifact_dir=ARTIFACT_DIR,
        new_embeddings=np.load(day_emb_paths[d - 1]),
    )
    checkpoints.append(result.manifest)
    for note in result.notes:
        print(' ', note)
    cov = result.manifest.coverage_cal
    ta  = result.manifest.teacher_agreement_cal
    print(f'  n_traces={result.manifest.n_traces:,}  '
          f'coverage={cov:.1%}  '
          f'TA={ta:.3f}  '
          f'method={result.manifest.selected_method}')

print('\nAll days complete.')

## 5 · Coverage & teacher-agreement growth

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

days         = list(range(1, N_DAYS + 1))
coverages    = [m.coverage_cal or 0.0 for m in checkpoints]
teacher_agr  = [m.teacher_agreement_cal or 0.0 for m in checkpoints]
n_traces_day = [m.n_traces for m in checkpoints]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Coverage
axes[0].plot(days, coverages, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Coverage (handled rate)')
axes[0].set_title('Coverage Growth')
axes[0].set_xticks(days)
axes[0].grid(True, alpha=0.3)

# Teacher agreement on handled traffic
axes[1].plot(days, teacher_agr, 's-', color='darkorange', linewidth=2, markersize=7)
axes[1].axhline(0.9, color='red', linestyle='--', alpha=0.5, label='target (0.90)')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Teacher Agreement')
axes[1].set_title('Teacher Agreement (calibration)')
axes[1].set_xticks(days)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Trace volume
axes[2].bar(days, n_traces_day, color='mediumseagreen', alpha=0.8)
axes[2].set_xlabel('Day')
axes[2].set_ylabel('Cumulative trace count')
axes[2].set_title('Training Data Volume')
axes[2].set_xticks(days)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(DATA_DIR / 'dynamic_coverage_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → dynamic_coverage_growth.png')

## 6 · Per-intent handled-rate over time

Which intents flip from *deferred* to *handled* as more traces arrive?  
We re-run the final router on the full training set and track per-label decisions.

In [ ]:
# Re-run the router on cumulative training data at each checkpoint
# We track per-intent handled rate by reloading the artifact after each day's fit.
# For efficiency we use the qualitative report's slice data (already computed).

# Load the last result's qualitative report for the temporal delta view
from tracer.policy.artifacts import load_qualitative_report

qual_path = ARTIFACT_DIR / 'qualitative_report.json'
if qual_path.exists():
    qr = load_qualitative_report(qual_path)
    print(f'Slices: {len(qr.slices)}')
    print(f'Temporal deltas: {len(qr.temporal_deltas)}')
else:
    print('No qualitative report found - skipping.')
    qr = None

In [ ]:
# Build per-intent handled rate by running each day's router on ALL traces seen so far
# We accumulate embeddings and labels day by day.

from tracer.runtime.router import Router

# Rebuild the per-day router snapshots using the same artifact dir.
# We only have the final router saved, so we estimate day-by-day deltas
# by computing handled rates on the *day's own data* via the final router.
# (A full per-day snapshot would require saving separate artifact dirs per day.)

router = _tracer.load_router(ARTIFACT_DIR)

intent_handled = defaultdict(list)  # label -> [handled_rate_day1, ..., handled_rate_dayN]

cumulative_texts  = []
cumulative_labels = []
cumulative_embs   = []

for d, (chunk, emb_path) in enumerate(zip(day_indices, day_emb_paths), 1):
    embs_d = np.load(emb_path)
    texts_d  = [train_texts[i]  for i in chunk]
    labels_d = [train_labels[i] for i in chunk]
    cumulative_texts  += texts_d
    cumulative_labels += labels_d
    cumulative_embs.append(embs_d)

    X_cum = np.vstack(cumulative_embs)

    # Route all cumulative samples
    per_label_handled  = defaultdict(list)
    per_label_deferred = defaultdict(list)
    for i, emb in enumerate(X_cum):
        out   = router.predict(emb)
        label = cumulative_labels[i]
        if out['decision'] == 'handled':
            per_label_handled[label].append(1)
        else:
            per_label_deferred[label].append(1)

    for label in label_names:
        n_h = len(per_label_handled[label])
        n_d = len(per_label_deferred[label])
        total = n_h + n_d
        intent_handled[label].append(n_h / total if total > 0 else 0.0)

    print(f'Day {d} routed - cumulative {len(cumulative_texts):,} samples')

print('Done.')

In [ ]:
# Sort intents by final-day handled rate (descending) and plot top/bottom 20
final_rates = {label: intent_handled[label][-1] for label in label_names}
sorted_labels = sorted(final_rates, key=lambda l: final_rates[l], reverse=True)

top20    = sorted_labels[:20]
bottom20 = sorted_labels[-20:]
plot_labels = top20 + bottom20

matrix = np.array([intent_handled[l] for l in plot_labels])  # (40, N_DAYS)

fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Handled rate', shrink=0.6)

ax.set_xticks(range(N_DAYS))
ax.set_xticklabels([f'Day {d}' for d in days])
ax.set_yticks(range(len(plot_labels)))
ax.set_yticklabels(plot_labels, fontsize=8)
ax.axhline(len(top20) - 0.5, color='black', linewidth=2)
ax.set_title('Per-intent handled rate over time\n(top 20 above line, bottom 20 below)', fontsize=12)

plt.tight_layout()
plt.savefig(DATA_DIR / 'dynamic_intent_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → dynamic_intent_heatmap.png')

## 7 · Test-set evaluation at each checkpoint

We evaluate the **final router** on the held-out test set.  
For day-by-day test metrics we would need to save separate artifact dirs; here we show the final checkpoint quality and a breakdown by intent.

In [ ]:
# Final-checkpoint test evaluation
from sklearn.metrics import f1_score

decisions_test = []
labels_test    = []
correct_test   = []

for emb, true_label in zip(test_embeddings, test_labels):
    out = router.predict(emb)
    decisions_test.append(out['decision'])
    labels_test.append(out['label'])
    correct_test.append(out['label'] == true_label)

handled_mask = np.array(decisions_test) == 'handled'
coverage_test   = handled_mask.mean()
accuracy_handled = np.array(correct_test)[handled_mask].mean() if handled_mask.any() else 0.0

# End-to-end F1: handled samples use router label; deferred samples use teacher (perfect)
e2e_preds = [
    labels_test[i] if decisions_test[i] == 'handled' else test_labels[i]
    for i in range(len(test_labels))
]
e2e_f1 = f1_score(test_labels, e2e_preds, average='macro')

print('── Test-set results (final checkpoint) ──────────────────────────')
print(f'Coverage      : {coverage_test:.1%}  ({handled_mask.sum():,} / {len(test_labels):,} handled)')
print(f'Accuracy (handled): {accuracy_handled:.3f}')
print(f'End-to-end macro F1: {e2e_f1:.3f}')
print(f'LLM calls saved: {coverage_test:.1%} of traffic')

In [ ]:
# Per-intent coverage on test set (handled rate)
intent_test_handled  = defaultdict(int)
intent_test_total    = defaultdict(int)
intent_test_correct  = defaultdict(int)

for emb, true_label in zip(test_embeddings, test_labels):
    out = router.predict(emb)
    intent_test_total[true_label] += 1
    if out['decision'] == 'handled':
        intent_test_handled[true_label] += 1
        if out['label'] == true_label:
            intent_test_correct[true_label] += 1

intent_stats = []
for label in label_names:
    total   = intent_test_total[label]
    handled = intent_test_handled[label]
    correct = intent_test_correct[label]
    intent_stats.append({
        'intent':   label,
        'n_test':   total,
        'coverage': handled / total if total > 0 else 0.0,
        'accuracy': correct / handled if handled > 0 else 0.0,
    })

intent_stats.sort(key=lambda x: x['coverage'], reverse=True)

print(f'{'Intent':<40} {'N':>5}  {'Cov':>6}  {'Acc':>6}')
print('-' * 62)
for s in intent_stats[:20]:
    print(f"{s['intent']:<40} {s['n_test']:>5}  {s['coverage']:>5.1%}  {s['accuracy']:>5.1%}")
print('...')
print('(bottom 20)')
for s in intent_stats[-20:]:
    print(f"{s['intent']:<40} {s['n_test']:>5}  {s['coverage']:>5.1%}  {s['accuracy']:>5.1%}")

In [ ]:
# Scatter: coverage vs accuracy per intent
covs  = [s['coverage'] for s in intent_stats]
accs  = [s['accuracy'] for s in intent_stats]
sizes = [s['n_test'] * 4 for s in intent_stats]

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(covs, accs, s=sizes, alpha=0.6, c=covs, cmap='viridis')
plt.colorbar(sc, ax=ax, label='Coverage')
ax.axvline(coverage_test, color='red', linestyle='--', alpha=0.5, label=f'mean cov {coverage_test:.1%}')
ax.set_xlabel('Handled rate (coverage)')
ax.set_ylabel('Accuracy on handled samples')
ax.set_title('Per-intent: Coverage vs Accuracy on Test Set')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / 'dynamic_intent_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → dynamic_intent_scatter.png')

## 8 · XAI report - final checkpoint

Inspect the qualitative report produced by the final `tracer.update()`.  
This covers four lenses: slice finder, representative examples, boundary pairs, and temporal deltas.

In [ ]:
from tracer.policy.artifacts import load_qualitative_report

qr_path = ARTIFACT_DIR / 'qualitative_report.json'
if not qr_path.exists():
    print('No qualitative report - refit produced no deployable pipeline.')
else:
    qr = load_qualitative_report(qr_path)
    print(qr.summary)
    print(f'Coverage: {qr.coverage:.1%}  |  TA(handled): {qr.teacher_agreement_handled:.3f}')
    print(f'Slices: {len(qr.slices)}  |  Examples: {len(qr.handled_examples)} handled, '
          f'{len(qr.deferred_examples)} deferred  |  Boundary pairs: {len(qr.boundary_pairs)}')

In [ ]:
if qr_path.exists():
    print(f'{'Slice':<45} {'N':>5}  {'Handled':>8}  {'Deferred':>9}  {'TA':>6}')
    print('-' * 80)
    for s in sorted(qr.slices, key=lambda x: x.handled_rate, reverse=True)[:20]:
        ta_str = f"{s.teacher_agreement_handled:.3f}" if s.teacher_agreement_handled is not None else ' n/a '
        print(f"{s.slice_name:<45} {s.count:>5}  {s.handled_rate:>7.1%}  {s.deferred_rate:>8.1%}  {ta_str:>6}")

In [ ]:
if qr_path.exists():
    print('── Handled examples (high-confidence) ───────────────────────────────')
    for ex in qr.handled_examples[:5]:
        score_str = f'{ex.accept_score:.3f}' if ex.accept_score is not None else 'n/a'
        print(f'  [{ex.teacher_label}] score={score_str}')
        print(f'  "{ex.input_preview}"')
        print()

    print('── Deferred examples (low-confidence) ───────────────────────────────')
    for ex in qr.deferred_examples[:5]:
        score_str = f'{ex.accept_score:.3f}' if ex.accept_score is not None else 'n/a'
        print(f'  [{ex.teacher_label}] score={score_str}')
        print(f'  "{ex.input_preview}"')
        print()

In [ ]:
if qr_path.exists() and qr.boundary_pairs:
    print('── Boundary pairs (same label, different routing) ────────────────────')
    for bp in qr.boundary_pairs[:5]:
        print(f'  Intent: {bp.teacher_label}')
        h_score = f'{bp.handled_score:.3f}' if bp.handled_score is not None else 'n/a'
        d_score = f'{bp.deferred_score:.3f}' if bp.deferred_score is not None else 'n/a'
        print(f'  HANDLED  (score={h_score}): "{bp.handled_preview}"')
        print(f'  DEFERRED (score={d_score}): "{bp.deferred_preview}"')
        print()

In [ ]:
if qr_path.exists() and qr.temporal_deltas:
    deltas_sorted = sorted(qr.temporal_deltas, key=lambda d: abs(d.delta), reverse=True)

    print(f'{'Intent':<40}  {'Prev':>6}  {'Curr':>6}  {'Δ':>7}')
    print('-' * 65)
    for td in deltas_sorted[:20]:
        arrow = '↑' if td.delta > 0 else '↓'
        print(f"{td.label:<40}  {td.previous_handled_rate:>5.1%}  {td.current_handled_rate:>5.1%}  "
              f"{arrow}{abs(td.delta):>5.1%}")
else:
    # Fall back to our own per-intent delta (day4 → day5)
    print('Temporal deltas from day 4 → day 5 (top movers):')
    own_deltas = []
    for label in label_names:
        rates = intent_handled[label]
        if len(rates) >= 2:
            own_deltas.append((label, rates[-2], rates[-1], rates[-1] - rates[-2]))
    own_deltas.sort(key=lambda x: abs(x[3]), reverse=True)
    print(f'{'Intent':<40}  {'Day N-1':>7}  {'Day N':>7}  {'Δ':>7}')
    print('-' * 65)
    for label, prev, curr, delta in own_deltas[:20]:
        arrow = '↑' if delta > 0 else '↓'
        print(f"{label:<40}  {prev:>6.1%}  {curr:>6.1%}  {arrow}{abs(delta):>5.1%}")

## Summary

| Metric | Value |
|--------|-------|
| Training days simulated | 5 |
| Total training traces | see Day 5 manifest |
| Final coverage (cal) | see `checkpoints[-1].coverage_cal` |
| Final TA (cal) | see `checkpoints[-1].teacher_agreement_cal` |
| Test coverage | see cell 7 |
| Test accuracy (handled) | see cell 7 |
| E2E macro-F1 | see cell 7 |

### Key takeaways

- Coverage grows monotonically as more traces arrive - the flywheel effect
- Intents with tight embedding clusters become self-serve earliest
- The conformal acceptor guarantees ≥ 90% teacher-agreement on handled traffic
- `tracer.update()` is a full refit on all historical data, not a delta - this ensures calibration stays valid

### Next steps

- Adjust `FitConfig.target_teacher_agreement` to trade coverage for precision
- Try embedding models with higher dimensionality for harder intents
- Add real teacher labels (from production LLM calls) to replace the ground-truth shortcut used here